<a href="https://colab.research.google.com/github/GauriNehe/Flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GauriNehe/Flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule (plain words): Flag pages that are old (created 180+ days ago) AND currently losing search visibility (impressions this month lower than 3 months ago). These are stale, declining pages worth a refresh review.
Reason codes it can output:
STALE_DECLINING — old page, impressions dropped
STALE_STABLE — old page, impressions flat/growing (lower priority)
FRESH — recently created, not yet eligible for refresh logic

In [ ]:
!pip install duckdb --quiet
import duckdb, os
import pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

q_rule_check = con.sql("""
    SELECT COUNT(*) AS eligible_pages
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    WHERE content_created_date <= DATE '2026-03-01' - INTERVAL 180 DAY
      AND is_published IS TRUE
""").df()
print(q_rule_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   eligible_pages
0          181959


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = 1 if STALE_DECLINING, weighted by how much impressions dropped. Rank descending by score, write to work/outputs/baseline_action_score.csv.

In [ ]:

df_perf = con.sql("""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    GROUP BY content_hash_id
""").df()

df_perf_dec = con.sql("""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_dec
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-12/data_0.parquet'
    GROUP BY content_hash_id
""").df()

df_content = con.sql("""
    SELECT content_hash_id, content_created_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    WHERE is_published IS TRUE
""").df()

merged = df_content.merge(df_perf, on="content_hash_id", how="left") \
                    .merge(df_perf_dec, on="content_hash_id", how="left")

merged["impressions_march"] = merged["impressions_march"].fillna(0)
merged["impressions_dec"] = merged["impressions_dec"].fillna(0)
merged["is_stale"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(merged["content_created_date"])).dt.days >= 180
merged["decline"] = merged["impressions_dec"] - merged["impressions_march"]

merged["reason_code"] = merged.apply(
    lambda r: "STALE_DECLINING" if r["is_stale"] and r["decline"] > 0
    else ("STALE_STABLE" if r["is_stale"] else "FRESH"), axis=1
)
merged["score"] = merged["decline"].clip(lower=0)
merged["action"] = merged["reason_code"].map({
    "STALE_DECLINING": "refresh_review",
    "STALE_STABLE": "monitor",
    "FRESH": "no_action"
})

ranked = merged.sort_values("score", ascending=False)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(ranked.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                 content_hash_id content_created_date  impressions_march  \
120837  content_469512b149928d3e           2025-10-29             1051.0   
160129  content_3508adb0f05ec0b9           2025-04-14            39836.0   
161335  content_5538d1fc8d19fa4a           2025-04-14            23756.0   
126090  content_99d1b38d4d5e03a3           2025-08-27              691.0   
88736   content_4d0d79fc12632ef8           2025-07-31            65736.0   
178305  content_df960beba08b53e3           2025-08-26            42092.0   
167375  content_f906d197db17fbc6           2025-10-22            16086.0   
249605  content_53031986a96b7913           2025-07-10                0.0   
160613  content_425a23ff83f9f448           2025-10-24            37408.0   
67439   content_017ff04acfbfccc9           2025-10-29             2321.0   

        impressions_dec  is_stale   decline      reason_code     score  \
120837         159006.0     False  157955.0            FRESH  157955.0   
160129         

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review: for stale/declining picks, confidence is high since both signals align. For FRESH picks appearing near the top (a rule weakness — see Section 4), confidence is low since the rule doesn't actually gate action by staleness, only by raw decline magnitude.

In [ ]:
top20 = ranked.head(20)[["content_hash_id", "action", "reason_code", "score", "is_stale"]].copy()
top20["confidence"] = top20["is_stale"].map({True: "high", False: "low"})
top20["would_be_wrong_if"] = top20.apply(
    lambda r: "the page is intentionally seasonal, not declining" if r["is_stale"]
    else "a fresh page has a huge score but no real refresh need — score isn't gated by staleness",
    axis=1
)
print(top20.to_string(index=False))

         content_hash_id         action     reason_code    score  is_stale confidence                                                                       would_be_wrong_if
content_469512b149928d3e      no_action           FRESH 157955.0     False        low a fresh page has a huge score but no real refresh need — score isn't gated by staleness
content_3508adb0f05ec0b9 refresh_review STALE_DECLINING  94476.0      True       high                                       the page is intentionally seasonal, not declining
content_5538d1fc8d19fa4a refresh_review STALE_DECLINING  86400.0      True       high                                       the page is intentionally seasonal, not declining
content_99d1b38d4d5e03a3 refresh_review STALE_DECLINING  73065.0      True       high                                       the page is intentionally seasonal, not declining
content_4d0d79fc12632ef8 refresh_review STALE_DECLINING  65812.0      True       high                                       the pa

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Rows 1, 6, 8, 9, 11 (and others marked FRESH/low confidence) are weak because the score is raw impression decline, not gated by staleness — a fresh page with a big swing outranks genuinely stale, declining pages that need review more urgently. This is a design flaw in the rule, not a data issue: the score formula should multiply by is_stale (or filter to stale-only) before ranking.
Leakage check: No product flags or future windows leaked in — decline only compares 2025-12 vs 2026-03, both fully in the past relative to any decision point, and no label-derived column (like a future outcome) was used as an input feature.

In [ ]:
future_check = con.sql("""
    SELECT MAX(report_date) AS latest_date_used
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()
print(future_check)
print("Confirms latest data used ends within March 2026, no leakage beyond the panel's window.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  latest_date_used
0       2026-03-31
Confirms latest data used ends within March 2026, no leakage beyond the panel's window.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.